# Matched training and prospective RGB audit

This notebook reproduces the locked follow-up protocol: five Fashion-MNIST families, two recipe arms and three paired training seeds, each trained for 60 epochs. Three checkpoint policies yield 90 evaluations from 30 trajectories. The primary comparison uses CNN at fixed epoch 60 on the previously inspected validation split. The separate completed RGB audit uses six frozen models and a prospectively specified image cohort.

Recorded results, environment identity and statistical interpretation are available in the accompanying report and machine-readable evidence. The four code cells preserve the executed experiment configuration.


## Runtime and source preparation

Mount persistent storage, validate the immutable source archive, install auxiliary dependencies and verify the A100 runtime. The archived source identity binds the protocol and implementation used by the experiment.


In [ ]:
import os, sys, json, hashlib, zipfile, subprocess
from pathlib import Path, PurePosixPath
from importlib.metadata import version
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
from google.colab import drive, files
import torch
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0), 'This experiment requires an A100 runtime.'
drive.mount('/content/drive')
SOURCE_SHA256 = 'a1d7a73a01ef749081516d432bbb3f839c134c70ee4fa0b655a87948ccc514bf'
STORE = Path('/content/drive/MyDrive/G-M10_Confirmatory_20261029')
STORE.mkdir(parents=True, exist_ok=True)
def file_sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
SOURCE = STORE/'source'/SOURCE_SHA256/'CO3133_Confirmatory_Source_20261029.zip'
SOURCE.parent.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    print('Source archive: CO3133_Confirmatory_Source_20261029.zip')
    uploaded = files.upload()
    matches = [b for b in uploaded.values() if hashlib.sha256(b).hexdigest() == SOURCE_SHA256]
    assert len(matches) == 1, 'The source archive does not match this notebook.'
    SOURCE.write_bytes(matches[0])
    del uploaded, matches
assert file_sha(SOURCE) == SOURCE_SHA256, 'Stored source identity differs.'
REPO = Path('/content/CO3133Confirm_'+SOURCE_SHA256[:12])
REPO.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE) as archive:
    manifest = json.loads(archive.read('SOURCE_MANIFEST.json'))
    paths = {p['path'] for p in manifest['payloads']} | {'SOURCE_MANIFEST.json'}
    assert len(paths) == len(archive.namelist()) and paths == set(archive.namelist())
    assert sum(p['bytes'] for p in manifest['payloads']) < 256*1024*1024
    for item in manifest['payloads']:
        path = PurePosixPath(item['path'])
        assert not path.is_absolute() and '..' not in path.parts and ':' not in str(path) and '\\' not in str(path)
        value = archive.read(str(path))
        assert len(value) == item['bytes'] and hashlib.sha256(value).hexdigest() == item['sha256']
        target = REPO.joinpath(*path.parts)
        assert target.resolve().is_relative_to(REPO.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert file_sha(target) == item['sha256'], 'Local source differs from the locked package.'
        else:
            target.write_bytes(value)
    (REPO/'SOURCE_MANIFEST.json').write_bytes(archive.read('SOURCE_MANIFEST.json'))
before = {p:version(p) for p in ('torch','torchvision')}
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-confirmatory-colab.txt')],check=True)
assert before == {p:version(p) for p in before}, 'CUDA torch/torchvision pair changed.'
caches = [Path('/content/drive/MyDrive/G-M10_GarmentLab_ImproveToday/cache'),
          Path('/content/drive/MyDrive/G-M10_GarmentLab_A100_Deadline/cache')]
CACHE = next((p for p in caches if (p/'train2020.zip').is_file()), STORE/'cache')
CACHE.mkdir(parents=True,exist_ok=True)
OUT = STORE/'runs'/('g_m10_confirmatory_run01_'+SOURCE_SHA256[:12])
command = [sys.executable,'-u',str(REPO/'tools/run_confirmatory.py'),
           '--out',str(OUT),'--cache',str(CACHE),
           '--data-dir','/content/fashion_confirmation_data','--device','cuda']
print('GPU:',torch.cuda.get_device_name(0))
print('Outputs:',OUT)
print('Raw cache:',CACHE)


## Protocol validation and software smoke checks

Validate task definitions, data identities and source bindings. The short forward/backward checks verify execution before the measured study.


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',
    'tests/test_a1confirm_study.py','tests/test_garmentlab_prospective.py',
    'tests/test_confirmatory_controller.py',
    '--basetemp','/content/confirmatory_tests_'+SOURCE_SHA256[:12]],cwd=REPO,check=True)
subprocess.run(command+['--stage','plan'],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','a1confirm','smoke','--out',str(OUT/'fashion_confirmation'),
    '--data-dir','/content/fashion_confirmation_data','--device','cuda'],cwd=REPO,check=True)
print('Software checks and A100 batch execution passed.')


## Matched study execution

Execute or resume committed trajectories under the recorded time budget. The controller retains all declared checkpoint policies and verifies the completed RGB evidence without fitting new RGB models.


In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
with (OUT/'NOTEBOOK_CONSOLE.log').open('a',encoding='utf-8') as log:
    process = subprocess.Popen(command+['--stage','all'],cwd=REPO,stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line); log.flush()
        returncode = process.wait()
    except BaseException:
        import signal
        process.send_signal(signal.SIGINT)
        try: process.wait(timeout=120)
        except subprocess.TimeoutExpired: process.terminate()
        raise
assert returncode == 0, 'Controller stopped with an error. Saved logs and partial evidence identify the failure.'


## Result export and integrity verification

Export task records, checkpoints and predictions, then verify payload hashes and completion status. Partial exports retain their incomplete status.


In [ ]:
subprocess.run(command+['--stage','export'],cwd=REPO,check=True)
ARCHIVE = OUT.parent/(OUT.name+'_results.zip')
with zipfile.ZipFile(ARCHIVE) as archive:
    assert archive.testzip() is None
    receipt = json.loads(archive.read('RESULT_MANIFEST.json'))
    for item in receipt['files']:
        value = archive.read(item['path'])
        assert len(value)==item['bytes'] and hashlib.sha256(value).hexdigest()==item['sha256']
    status = json.loads(archive.read('RESULT_STATUS.json'))
print('STATUS:',status['status'])
print('FASHION:',status['fashion'].get('completed_trajectories',0),'/30')
print('RGB:',status['rgb_status'])
print('RESULT:',ARCHIVE)
print('SHA256:',file_sha(ARCHIVE))
files.download(str(ARCHIVE))
